# v2 SIM — 개인 의도 보정과 XYZ 배율

로봇 연결 없이 Quest → 보정 → 배율 → 기존 IK → 가상 관절 피드백을 확인합니다. SIM은 이상적인 관절 추종 모델이며 모터·접촉·네트워크 지연을 재현하지 않습니다.

위치 매핑: `operator delta → C → XYZ gains → 20cm 작업범위 → IK`. Z를 고정하거나 전진 궤적으로 강제 투영하지 않습니다.
기준 자세는 로봇의 현재 EEF와 편한 컨트롤러 자세를 양손 grip 시점에 대응시킵니다. 좌표는 로봇 전방 +X, 왼쪽 +Y, 위 +Z입니다.


**자세 유지 IK 적용:** 첫 양손 grip 조작의 실측 자세를 기준으로 유지합니다(DRY RUN 포함). 위치·방향 허용오차는 1 mm·0.5°이며, grip을 다시 잡아도 자세 기준은 유지됩니다. 기존 IK 비교는 `POSTURE_IK_ENABLED=False`로 바꾸고 런타임을 다시 시작하세요. [설정·테스트 안내](../docs/POSTURE_IK.md).


수동 관절 조정: 조작을 멈춘 뒤 관절과 절대 목표 각도(°)를 선택하고 **관절 미리보기 → 조정 실행**을 누릅니다. 같은 창의 **Cancel return**으로 중단하고, 조정 후 **Initial pose**로 복귀할 수 있습니다. 최초 HOME 목표는 보존됩니다.
그리퍼는 기본 60°/s, 새 bridge 사용 시 10~120°/s를 입력할 수 있습니다. REAL 적용 전 로봇 PC의 `inbound.py`와 `fixed_quest_protocol.py`를 동일 버전으로 교체하고 inbound/outbound를 재시작하세요. HOME과 수동 조정은 최대 10°/s입니다.


In [1]:
# Locate this standalone workspace from a notebook or repository directory.
from pathlib import Path
import runpy

_v2_candidates = [candidate for base in (Path.cwd(), *Path.cwd().parents)
                  for candidate in (base, base / "VR_teleoperation")]
_v2_root = next((candidate for candidate in _v2_candidates
                 if (candidate / ".project_v2").is_file()), None)
if _v2_root is None:
    raise RuntimeError("Open this notebook from the VR_teleoperation workspace.")
_v2_bootstrap = runpy.run_path(str(_v2_root / "package" / "init_project.py"))
for _v2_name in _v2_bootstrap["__all__"]:
    globals()[_v2_name] = _v2_bootstrap[_v2_name]


import numpy as np



[VR_teleoperation / ffw_sg2_vr_teleoperation] initialized.
 - PROJECT_DIR: /home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation


## 설정 — v1 후보 SIM 시험

현재 v1.json은 궤적 검증 미통과 후보입니다. 원본 accepted는 바꾸지 않고, SIM 전용 명시적 옵션으로 원본 시연과 보정 재계산을 확인한 뒤 불러옵니다. REAL에서는 이 옵션을 거절합니다.

같은 본인·Quest 방향·작업 위치를 확인하세요. 프로필은 실행 시작 시 읽으며 다른 값을 쓰려면 STOP 후 설정부터 다시 실행합니다. 일반 통과 프로필로 돌아가려면 ALLOW_FAILED_PROFILE_IN_SIM=False로 바꾸고 PROFILE_PATH를 지정하세요. 보정 없이 비교하려면 해당 옵션을 False로 하고 PROFILE_PATH=None으로 설정하세요.


In [2]:
from package.intent_teleop import IntentTeleopSettings, selected_profile, build_models, run_intent_teleop, tracking_mode_text
settings = IntentTeleopSettings(
    backend="sim",
    PROFILE_PATH=PROJECT_DIR / "calibration/diagnostics/v1.json",
    SIM_CANDIDATE_CAPTURE_PATH=PROJECT_DIR / "calibration/diagnostics/v1_trajectory.json",
    ALLOW_FAILED_PROFILE_IN_SIM=True,  # 검증 실패 후보를 명시적으로 SIM에서만 시험
    CONFIRM_CALIBRATION_SETUP=True,  # 본인·Quest 방향·작업 위치를 확인한 후 True
    POSITION_SCALE=.5, POSITION_SCALE_Y=.5, POSITION_SCALE_Z=.5,
    MAX_JOINT_SPEED=np.deg2rad(60.),  # 실행 창에서 10~120 숫자 입력 + 적용
    TELEOP_RAMP_S=3.0,  # 시작/재그립 후 팔 속도 상한까지 ramp 시간 [초]
    MAX_GRIPPER_SPEED=np.deg2rad(60.),  # 그리퍼 전용 상한. 새 worker에서 10~120 숫자 입력 + 적용 (복귀 중 10°/s)
    IK_RECOVERY_TIMEOUT_S=1.0,  # IK 미수렴 시 최대 1초 유지·재시도 (0: 즉시 중단)
    NULL_REFERENCE="self_motion",  # 기존 IK 비교 모드에서만 사용
    POSTURE_IK_ENABLED=True,  # False: 기존 IK로 비교 (런타임 재시작)
    TELEOP_COLLISION_CHECK=False,  # Teleoperation OFF · Initial pose/Reset pose 충돌 검사 ON
    POSTURE_POSITION_TOLERANCE_M=.001,  # IK 후보 허용오차 1 mm
    POSTURE_ROTATION_TOLERANCE_DEG=.5,  # IK 후보 허용오차 0.5°
    POSTURE_ELBOW_LIMIT_DEG=85.,  # 초기 팔꿈치 방향에서 반대편으로 넘어가지 않음
    NULL_ANCHOR_ON_GRIP=False,  # 초기 팔 자세 기준은 첫 조작 시 실측으로 한 번만 저장
    QUEST_HOST=None,  # 자동 LAN IP가 틀리면 PC IP를 문자열로 입력
    TRACKING_PAUSE_S=.1, ACCEPT_EMULATED_POSE=True,  # 시작 모드: 추정 위치 사용 (컨트롤러가 사라지면 0.1초 뒤 정지)
    TRACKING_MODE_BUTTONS=("emulated", "pause_0.5"),  # 실행 창 버튼: 추정 위치 사용 / 정지 0.5초
    DRY_RUN=True,
)
settings.validate()
print("IK 모드:", "초기 팔 자세 유지" if settings.POSTURE_IK_ENABLED else "기존 IK")
profile, matrices = selected_profile(settings)
print("개인 보정:", "미적용 — identity" if profile is None else profile["profile_id"])
print("XYZ:", settings.POSITION_SCALES["left"], "| 속도:", np.rad2deg(settings.MAX_JOINT_SPEED),
      "| 그리퍼:", np.rad2deg(settings.MAX_GRIPPER_SPEED), "| 기존 IK null 기준:", settings.NULL_REFERENCE)
print("SIM 후보 시험:", settings.ALLOW_FAILED_PROFILE_IN_SIM, "| 원본 accepted:", None if profile is None else profile["accepted"])
print("추적 손실 시작 모드:", tracking_mode_text({"tracking_pause_s": settings.TRACKING_PAUSE_S,
                                                 "accept_emulated": settings.ACCEPT_EMULATED_POSE}))


개인 보정: 51738a755a527d0c
XYZ: [0.5, 0.5, 0.5] | 속도: 59.99999999999999 | 그리퍼: 59.99999999999999 | null 기준: self_motion
SIM 후보 시험: True | 원본 accepted: False
추적 손실 시작 모드: 추정 위치 사용 (완전 소실 시 정지 0.1초)


## 모델 준비 확인 (통신 없음)

기존 FK·IK와 관절 범위 검사를 사용합니다. 충돌 검사는 Teleoperation에서 OFF, Initial pose·Reset pose·수동 조정에서 ON입니다. 아래 셀은 모델만 확인합니다.


In [3]:
models = build_models(settings)
from utils import get_sg2_wrist_control_T
for hand in ("left", "right"):
    print(hand, "EEF (m):", get_sg2_wrist_control_T(models["env"], hand)[:3, 3])
del models


MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
Command joints: 19 | Collision pairs: 499
No robot connection or commands in this setup cell.
left EEF (m): [0.468004 0.2275   1.100596]
right EEF (m): [ 0.468004 -0.2275    1.100596]


## 실행

bringup/outbound/inbound는 실행하지 않아도 됩니다.

- 다른 Quest 수신 노트북을 STOP하여 8443 포트를 비우세요.
- 양손 grip으로 시작하고, 한쪽이라도 놓으면 양팔이 멈춥니다.
- XYZ는 grip을 모두 놓고 숫자 입력 후 적용합니다. 새 grip 기준에서 반영되므로 목표가 갑자기 바뀌지 않습니다.
- 속도는 실행 중 입력 후 적용할 수 있습니다. 감속 시 이전 명령 ACK를 기다릴 수 있습니다.
- 그리퍼 속도는 팔 속도와 별개로 실행 중 10~60°/s로 적용할 수 있습니다. 로봇 bridge가 그리퍼를 60°/s로 제한하므로 그 이상은 입력할 수 없고, Initial pose 복귀 중에는 복귀 속도를 따릅니다.
- Initial pose는 Quest/그립 없이 고정된 `worker_bringup` 자세(`BRINGUP_POSE`)로 10°/s 복귀합니다 — 실행 시작 시 어떤 자세였는지와 무관합니다. 머리는 `head_joint1`이 0.3 rad(약 17°) 아래를 향하고, 좌우 회전(`head_joint2`)만 실측값을 유지합니다.
- 누르는 순간 복귀 전체 경로를 기존 관절·5mm 충돌 검사로 확인합니다. 직선 경로가 막히면 J3·J5 먼저 풀기 → 전완·손목 먼저 풀기 → 오른팔 먼저 → 왼팔 먼저 순서로 단계 경로를 찾고, 명령은 확인한 경로 위로만 보냅니다. 모든 경로가 막히면 아무 명령도 보내지 않으므로, VR 조작으로 팔 사이를 벌린 뒤 다시 누르세요. 중간에 멈춘 복귀도 다시 누르면 현재 자세에서 새로 검사합니다.
- **추적 손실 모드**: 기본은 `추정 위치 사용`입니다. 한 손의 광학 추적이 끊겨도 Quest 추정 위치로 조작을 이어가고, 컨트롤러가 아예 사라지면 0.1초 뒤 정지합니다. 추정 위치는 손실 순간 수 cm~십수 cm 튈 수 있습니다.
- 실행 창에서 `정지 0.5초`로 바꾸면 추적 손실 중 현재 명령을 최대 0.5초 유지하고, 그 안에 추적이 돌아오면 같은 grip 기준에서 이어갑니다. 모드 전환은 양손 grip을 놓은 상태에서 적용됩니다.
- 두 모드 모두 한 프레임 20cm 초과 점프·grip 해제·피드백/실행 오류는 정지하고, grip당 20cm 작업범위와 속도 제한은 유지합니다. 두 추적 모드 모두 Teleoperation 충돌 검사는 OFF이며 자동 복귀에서는 ON입니다.
- 일시적인 IK 미수렴은 양팔·그리퍼의 현재 명령을 최대 1초 유지하며 최신 목표로 재시도합니다. 복구되면 같은 grip 기준에서 조작을 이어갑니다. 화면에 경과 시간과 재시도 횟수가 표시되며, 시간 초과·그립 해제·피드백 이상은 정지합니다.
- 팔꿈치 양옆의 두 회전(Joint 3·5)이 조작과 무관하게 감기지 않도록, EEF가 볼 수 없는 자세 성분만 시작 자세 쪽으로 유지합니다. 목표 추종에 쓰이는 성분은 기존과 같이 IK가 결정합니다.
- 로그는 `.cache/fixed_quest/recordings/*-intent-sim.jsonl`에 기록됩니다.

### v1 확인 순서

화면에 ‘SIM 후보 시험’과 v1 profile ID가 표시되는지 확인합니다. 편한 시작 자세에서 양손 grip을 잡고, 전진 → 좌우 → 상승 → 좌우＋상승 순서로 작은 움직임부터 확인합니다. 처음에는 XYZ를 모두 0.5로 유지하고, 전진 시 높이 변화 및 대각선 이동 시 앞뒤 이동이 얼마나 느껴지는지 확인하세요. 오류/멈춤이 있으면 종료 후 생성된 intent-sim.jsonl 로그로 분석합니다.

In [4]:
log_path = run_intent_teleop(settings)


MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
Command joints: 19 | Collision pairs: 499
No robot connection or commands in this setup cell.
Quest Browser: https://192.168.6.100:8443/
SIM | 양손 grip으로 시작 · 한쪽이라도 놓으면 양팔 정지
종료: PC_STOP_or_control_window_closed | Log: /home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperat

## 선택: 로봇·Quest 없이 자동 검증

실행 셀을 STOP한 뒤 실행합니다. 합성 시연의 축 보정과 기존 IK·protocol 흐름을 검사합니다. 개인 보정의 실험 결과가 아닙니다.


In [5]:
import runpy
validator = runpy.run_path(str(PROJECT_DIR / "scripts/validate_intent_pipeline.py"))
report = validator["run_validation"]()
print(report)
assert report["passed"], report
from IPython.display import Image, display
display(Image(filename=str(PROJECT_DIR / ".cache/validation/intent/trajectory.png")))



[VR_teleoperation / ffw_sg2_vr_teleoperation] initialized.
 - PROJECT_DIR: /home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/.cache/fixed_quest/intent_sim_scene.xml].
Command joints: 19 | Collision pairs: 499
No robot connection or commands in this setup cell.


KeyboardInterrupt: 